In [ ]:
.root <- getwd()
while (!file.exists(file.path(.root, 'CLAUDE.md')) && dirname(.root) != .root)
  .root <- dirname(.root)
setwd(.root)
rm(.root)

suppressPackageStartupMessages({
  library(tidyverse)
  library(arrow)
  library(ggthemes)
  library(scales)
  library(patchwork)
})

options(repr.plot.width = 14, repr.plot.height = 8)

BASE_THEME <- theme_fivethirtyeight() +
  theme(
    axis.title        = element_text(size = 14),
    axis.text         = element_text(size = 13),
    panel.background  = element_rect(fill = 'white', color = NA),
    plot.background   = element_rect(fill = 'white', color = NA),
    legend.background = element_rect(fill = 'white', color = NA),
    legend.text       = element_text(size = 12),
    legend.title      = element_text(size = 13),
    strip.text        = element_text(size = 14, face = 'bold'),
    plot.title        = element_text(size = 20, face = 'bold'),
    plot.subtitle     = element_text(size = 13)
  )
theme_set(BASE_THEME)

dir.create('results/plots/results', recursive = TRUE, showWarnings = FALSE)

In [ ]:
xrv <- read_parquet('results/xrv_causal.parquet') |>
  select(disruption_tax, distortion_tax, selection_tax,
         spatial_distortion_tax, distortion_share) |>
  filter(!is.na(distortion_tax), !is.na(selection_tax)) |>
  mutate(
    angular_disruption        = disruption_tax - spatial_distortion_tax,
    angular_distortion_share  = if_else(
      abs(angular_disruption) > 1e-6,
      pmax(0, pmin(1, (distortion_tax - spatial_distortion_tax) / angular_disruption)),
      NA_real_
    )
  )

cat(sprintf(
  'Loaded %s swings | median distortion_share = %.2f | median angular_distortion_share = %.2f\n',
  formatC(nrow(xrv), format = 'd', big.mark = ','),
  median(xrv$distortion_share, na.rm = TRUE),
  median(xrv$angular_distortion_share, na.rm = TRUE)
))

## Angular Disruption Attribution: Selection vs Distortion

In [ ]:
share_df <- xrv |>
  filter(!is.na(angular_distortion_share), abs(angular_disruption) > 1e-4)

pct_distortion_dom <- mean(share_df$angular_distortion_share >= 0.5) * 100
pct_selection_dom  <- 100 - pct_distortion_dom
mean_share         <- mean(share_df$angular_distortion_share)

p_share <- ggplot(share_df, aes(x = angular_distortion_share)) +
  geom_histogram(
    aes(fill = after_stat(x) >= 0.5),
    binwidth = 0.025, boundary = 0
  ) +
  scale_fill_manual(
    values = c('FALSE' = '#3498db', 'TRUE' = '#e74c3c'),
    labels = c('FALSE' = 'Selection-dominant  (< 50% distortion)',
               'TRUE'  = 'Distortion-dominant (>= 50% distortion)'),
    name = NULL
  ) +
  geom_vline(xintercept = 0.5,        color = 'black',   linewidth = 0.9, linetype = 'dashed') +
  geom_vline(xintercept = mean_share, color = '#f39c12', linewidth = 1.1) +
  annotate('text', x = mean_share + 0.015, y = Inf, vjust = 1.4,
           label = sprintf('Mean = %.2f', mean_share),
           color = '#f39c12', size = 4.5, fontface = 'bold') +
  annotate('text', x = 0.25, y = Inf, vjust = 2.5,
           label = sprintf('%.0f%% of swings\nselection-dominant', pct_selection_dom),
           color = '#2980b9', size = 5, fontface = 'bold') +
  annotate('text', x = 0.75, y = Inf, vjust = 2.5,
           label = sprintf('%.0f%% of swings\ndistortion-dominant', pct_distortion_dom),
           color = '#c0392b', size = 5, fontface = 'bold') +
  scale_x_continuous(
    labels = percent_format(accuracy = 1),
    expand = expansion(mult = c(0.01, 0.01)),
    limits = c(0, 1)
  ) +
  scale_y_continuous(labels = comma_format()) +
  labs(
    title    = 'Angular Disruption Attribution: Selection vs Distortion',
    subtitle = paste0(
      'Per-swing share of angular deviation explained by post-commit pitch movement  |  ',
      formatC(nrow(share_df), format = 'd', big.mark = ','), ' swings\n',
      '0% = all batter error  |  100% = all late movement  |  Orange line = mean'
    ),
    x = 'Angular distortion share (% of angular deviation from post-commit movement)',
    y = 'Number of swings'
  ) +
  theme(legend.position = 'top')

options(repr.plot.width = 14, repr.plot.height = 7)
p_share
ggsave('results/plots/results/angular_distortion_share.png', p_share,
       width = 14, height = 7, dpi = 150)